# Expert-CFG: VQA-RAD 5-Run Ablation (논문 Table 1 재현)

| Run | 설명 | γ | LoRA | RAG | Attn | Embed | Selective |
|-----|------|---|------|-----|------|-------|-----------|
| 1. baseline | Phi3.5V-Med (no RAG) | 1.0 | ✅ | ✗ | ✗ | ✗ | ✗ |
| 2. w/ RAG | 전체 샘플 RAG | 1.0 | ✅ | ✅ | ✗ | ✗ | ✗ |
| 3. w/ 5% RAG | 상위 5% PE 샘플만 RAG | 1.0 | ✅ | ✅ | ✗ | ✗ | ✅ 5% |
| 4. w/ Expert-RAG | 5% + attn+embed (γ=1.0) | 1.0 | ✅ | ✅ | ✅ | ✅ | ✅ 5% |
| 5. w/ 5% Expert-CFG | 5% + full CFG (γ=1.5) ← **논문 메인** | 1.5 | ✅ | ✅ | ✅ | ✅ | ✅ 5% |

**논문 Table 1 목표치 (VQA-RAD):**
- baseline: Open=69.27%, Closed=90.44%, Overall=82.04%
- 5% Expert-CFG: Open=74.86%, Closed=91.18%, Overall=84.70%

**필요한 Kaggle 데이터셋:**
```
/kaggle/input/expert-cfg-data/
├── test.json          # annotation + top_k_captions + highlights
└── VQA-RAD/images/

/kaggle/input/datasets/ciderchicken/expert-cfg-weights/
├── Phi-3.5-vision-instruct/   # 베이스 모델 가중치
└── lora_weights/
    └── logs_phi35_pubmed_instruct/   # Phi3.5V-Med LoRA
```

## Step 1. 패키지 설치

In [ ]:
!pip install -q \
    "transformers==4.48.3" \
    "accelerate==0.30.0" \
    "peft==0.11.1" \
    bitsandbytes

import importlib
import transformers, tokenizers
importlib.reload(transformers)
print("transformers:", transformers.__version__)   # 4.48.3
print("tokenizers  :", tokenizers.__version__)     # 0.21.x

## Step 2. Expert-CFG 클론

In [ ]:
import os, subprocess, sys, json, shutil

REPO_DIR = '/kaggle/working/Expert-CFG'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone', '--branch', 'feat/vqa-rad',
    'https://github.com/minseongkim-dev/expert-cfg',
    REPO_DIR,
], check=True)
print('클론 완료')

## Step 3. 모델 가중치 복사

레포 안의 `Phi-3.5-vision-instruct/`에는 커스텀 processor(`processing_phi3_v_cfg.py`)가 있음.  
Kaggle 데이터셋의 가중치(.safetensors 등)를 그 폴더로 복사해서 합침.

In [ ]:
import shutil

WEIGHTS_SRC = '/kaggle/input/datasets/ciderchicken/expert-cfg-weights/Phi-3.5-vision-instruct'
MODEL_DIR   = '/kaggle/working/Expert-CFG/Phi-3.5-vision-instruct'

copied = 0
for fname in os.listdir(WEIGHTS_SRC):
    src = os.path.join(WEIGHTS_SRC, fname)
    dst = os.path.join(MODEL_DIR, fname)
    if os.path.isfile(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'복사: {fname}')
        copied += 1

print(f'완료: {copied}개 파일 복사')

In [ ]:
# 호환성 패치
import json, shutil

# --- HF 모듈 캐시 초기화 ---
modules_cache = os.path.expanduser('~/.cache/huggingface/modules/transformers_modules')
if os.path.exists(modules_cache):
    shutil.rmtree(modules_cache)
    print('✅ transformers_modules 캐시 초기화')

# --- tokenizer_config.json AutoProcessor 교정 ---
# transformers가 tokenizer_config.json을 processor_config.json 이후에 읽어 덮어쓰므로 수정
tok_cfg_path = os.path.join(MODEL_DIR, 'tokenizer_config.json')
if os.path.exists(tok_cfg_path):
    with open(tok_cfg_path) as f:
        tok_cfg = json.load(f)
    old = tok_cfg.get('auto_map', {}).get('AutoProcessor', '')
    if old != 'processing_phi3_v_cfg.Phi3VProcessor':
        tok_cfg.setdefault('auto_map', {})['AutoProcessor'] = 'processing_phi3_v_cfg.Phi3VProcessor'
        with open(tok_cfg_path, 'w') as f:
            json.dump(tok_cfg, f, indent=2)
        print(f'✅ tokenizer_config.json AutoProcessor 교정: {old!r} → processing_phi3_v_cfg.Phi3VProcessor')
    else:
        print('ℹ️  tokenizer_config.json 이미 올바름')
else:
    print('⚠️  tokenizer_config.json 없음 — 건너뜀')

## Step 4. 경로 확인

In [ ]:
DATA_ROOT = '/kaggle/input/expert-cfg-data'
TEST_JSON = f'{DATA_ROOT}/test.json'
IMG_ROOT  = f'{DATA_ROOT}/VQA-RAD/images'
WORK_DIR  = '/kaggle/working'
LORA_DIR  = '/kaggle/input/datasets/ciderchicken/expert-cfg-weights/lora_weights/logs_phi35_pubmed_instruct'

for label, path in [
    ('model', MODEL_DIR),
    ('test.json', TEST_JSON),
    ('images/', IMG_ROOT),
    ('lora weights', LORA_DIR),
]:
    print(f"{('✅' if os.path.exists(path) else '❌')} {label}: {path}")

if os.path.exists(IMG_ROOT):
    print(f'   이미지 수: {len([f for f in os.listdir(IMG_ROOT) if f.endswith(".jpg")])}개')

## Smoke Test — RAG·CFG 효과 빠른 검증

> Full run(Step 5) 전에 모델이 제대로 동작하는지, RAG와 CFG가 실제로 효과가 있는지 확인합니다.
> 샘플 2개로 ~5분 소요.

| Test | 질문 유형 | 목표 |
|------|-----------|------|
| Smoke 1 | CLOSED (yes/no) | Baseline 실패 → RAG 정정 → CFG 확률 ↑ |
| Smoke 2 | OPEN (키워드 포함) | 64토큰에서 GT 키워드 생성 여부 |

In [ ]:
import json, os, subprocess, sys

REPO     = '/kaggle/working/Expert-CFG'
EX_DIR   = f'{REPO}/examples'
IMG_EX   = f'{EX_DIR}/images'

# examples/input_queries.json → smoke_one.json (CLOSED), smoke_two.json (OPEN)
with open(f'{EX_DIR}/input_queries.json') as f:
    examples = json.load(f)

with open(f'{EX_DIR}/smoke_one.json', 'w') as f:
    json.dump(examples[:1], f, indent=2)
with open(f'{EX_DIR}/smoke_two.json', 'w') as f:
    json.dump(examples[1:2], f, indent=2)

print(f"Smoke 1 | Q: {examples[0]['question']!r}  GT: {examples[0]['answer']!r}")
print(f"Smoke 2 | Q: {examples[1]['question']!r}  GT: {examples[1]['answer']!r}")

# ── Smoke Test 1: CLOSED 질문, max_new_tokens=32 ─────────────────────────────
EXPERIMENTS = [
    ('baseline',   ['--no_rag', '--cfg', '1.0']),
    ('rag',        ['--cfg', '1.0']),
    ('expert_cfg', ['--cfg', '1.5', '--use_attn_patch', '--use_embed_perturb']),
]

for name, extra in EXPERIMENTS:
    cmd = [
        sys.executable, 'demo.py',
        '--model_name_or_path', MODEL_DIR,
        '--use_qlora', '--output_dir', LORA_DIR,
        '--bf16', '--num_crops', '1',
        '--dataset', 'vqa-rad',
        '--input_json', f'{EX_DIR}/smoke_one.json',
        '--img_root',   IMG_EX,
        '--save_path',  f'{EX_DIR}/smoke1_{name}.json',
        *extra,
    ]
    r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    status = '✅' if r.returncode == 0 else '❌'
    print(f'{status} {name}: exit={r.returncode}')
    if r.returncode != 0:
        print(r.stderr[-600:])

In [ ]:
import re

def tokenize(text):
    return re.findall(r'[a-z0-9]+', str(text).lower())

def score_answer(gt, pred, answer_type='OPEN'):
    gt = str(gt).strip().lower()
    toks = tokenize(pred)
    if gt in ('yes', 'no') or answer_type.upper() == 'CLOSED':
        label = next((t for t in toks if t in ('yes', 'no')), None)
        return float(label == gt)
    gt_toks = tokenize(gt)
    return sum(t in toks for t in gt_toks) / len(gt_toks) if gt_toks else 0.0

# ── Smoke Test 1 결과 표시 ────────────────────────────────────────────────────
print("=" * 65)
print("[ Smoke Test 1 ]  CLOSED (yes/no) 질문")
print("=" * 65)

for name in ['baseline', 'rag', 'expert_cfg']:
    path = f'{EX_DIR}/smoke1_{name}.json'
    if not os.path.exists(path):
        print(f"  {name}: 결과 없음 (실행 실패)")
        continue
    with open(path) as f:
        data = json.load(f)
    r = data['results'][0] if 'results' in data else data[0]
    s = score_answer(r['answer'], r['prediction'], r.get('answer_type', 'CLOSED'))
    mark = '✓' if s == 1.0 else '✗'
    # 첫 yes/no 토큰 확률 출력
    yn_tok_prob = None
    for tok, prob in zip(r.get('token_preds', []), r.get('token_probs', [])):
        if tok.lower().strip() in ('yes', 'no'):
            yn_tok_prob = prob
            break
    prob_str = f'  (1st yes/no prob: {yn_tok_prob:.4f})' if yn_tok_prob else ''
    print(f"  [{name:<11}]  {mark}  GT={r['answer']!r:5}  Pred={r['prediction'][:60]!r}{prob_str}")

print()
print("기대: baseline=✗, rag=✓, expert_cfg=✓  그리고 CFG > RAG의 yes/no 확률")

In [ ]:
# ── Smoke Test 2: OPEN 질문, max_new_tokens=64 ───────────────────────────────
for name, extra in EXPERIMENTS:
    cmd = [
        sys.executable, 'demo.py',
        '--model_name_or_path', MODEL_DIR,
        '--use_qlora', '--output_dir', LORA_DIR,
        '--bf16', '--num_crops', '1',
        '--dataset', 'vqa-rad',
        '--input_json', f'{EX_DIR}/smoke_two.json',
        '--img_root',   IMG_EX,
        '--save_path',  f'{EX_DIR}/smoke2_{name}.json',
        *extra,
    ]
    r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    status = '✅' if r.returncode == 0 else '❌'
    print(f'{status} {name}: exit={r.returncode}')
    if r.returncode != 0:
        print(r.stderr[-600:])

# ── Smoke Test 2 결과 표시 ────────────────────────────────────────────────────
print()
print("=" * 65)
print("[ Smoke Test 2 ]  OPEN 질문 (GT 키워드 포함 여부)")
print("=" * 65)

for name in ['baseline', 'rag', 'expert_cfg']:
    path = f'{EX_DIR}/smoke2_{name}.json'
    if not os.path.exists(path):
        print(f"  {name}: 결과 없음")
        continue
    with open(path) as f:
        data = json.load(f)
    r = data['results'][0] if 'results' in data else data[0]
    s = score_answer(r['answer'], r['prediction'], 'OPEN')
    gt_toks = tokenize(r['answer'])
    pred_l  = r['prediction'].lower()
    found   = [t for t in gt_toks if t in pred_l]
    print(f"  [{name:<11}]  recall={s:.2f}  GT={r['answer']!r}")
    print(f"               found={found}  Pred={r['prediction'][:80]!r}")

print()
print("기대: rag/expert_cfg가 GT 키워드를 더 많이 포함")

In [ ]:
# ── Smoke Test 3: CFG가 토큰 확률을 어떻게 바꾸는지 비교 (PPT용) ─────────────
print("=" * 65)
print("[ Smoke Test 3 ]  RAG vs Expert-CFG 토큰 확률 비교")
print("=" * 65)

def load_first_result(path):
    with open(path) as f:
        data = json.load(f)
    return data['results'][0] if isinstance(data, dict) and 'results' in data else data[0]

rag_path = f'{EX_DIR}/smoke1_rag.json'
cfg_path = f'{EX_DIR}/smoke1_expert_cfg.json'

if os.path.exists(rag_path) and os.path.exists(cfg_path):
    rag_r = load_first_result(rag_path)
    cfg_r = load_first_result(cfg_path)

    rag_toks  = rag_r.get('token_preds', [])
    rag_probs = rag_r.get('token_probs', [])
    cfg_toks  = cfg_r.get('token_preds', [])
    cfg_probs = cfg_r.get('token_probs', [])

    n = min(len(rag_toks), len(cfg_toks), 12)
    print(f"\n  {'Token':<12} {'RAG':>7} {'CFG':>7} {'Δ':>8}")
    print(f"  {'-'*40}")
    for i in range(n):
        tok   = rag_toks[i]
        p_rag = rag_probs[i]
        p_cfg = cfg_probs[i] if i < len(cfg_probs) else 0.0
        diff  = p_cfg - p_rag
        arrow = ' ↑' if diff > 0.01 else (' ↓' if diff < -0.01 else '  ')
        print(f"  {tok!r:<12} {p_rag:>6.4f}  {p_cfg:>6.4f}  {diff:>+7.4f}{arrow}")

    print(f"\n  RAG  pred : {rag_r['prediction'][:70]!r}")
    print(f"  CFG  pred : {cfg_r['prediction'][:70]!r}")
    print(f"  Ground truth: {rag_r['answer']!r}")
    print()
    print("CFG는 조건부/비조건부 logit 차이를 키워 확신 있는 토큰의 확률을 높임")
else:
    print("  smoke1 결과 파일 없음 — 위 셀 먼저 실행하세요")

## Step 5. 5가지 실행 (논문 Table 1 순서)

> **실행 순서 중요**: Run 1 → Run 2 → (Run 3/4/5는 Run 1 결과가 필요).
> 각 셀 약 1-2시간 소요.

In [ ]:
# Run 1: Phi3.5V-Med baseline (LoRA, no RAG, γ=1.0)
# → 논문 Table 1 첫 번째 행: Open=69.27%, Closed=90.44%, Overall=82.04% 목표
cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '1',
    '--cfg',                '1.0',
    '--no_rag',
    '--save_path',          f'{WORK_DIR}/result_1_baseline.json',
]
print('▶ Run 1: Phi3.5V-Med baseline (no RAG, γ=1.0, LoRA)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 2: w/ RAG — 전체 샘플 RAG, γ=1.0 (CFG 없음)
cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '1',
    '--cfg',                '1.0',
    '--save_path',          f'{WORK_DIR}/result_2_rag_all.json',
]
print('▶ Run 2: w/ RAG (all samples, γ=1.0, LoRA)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 3: w/ 5% RAG — 상위 5% PE 샘플만 RAG 적용 (selective_rerun.py)
# Phase 1 결과(result_1_baseline.json)에서 PE 상위 5% 선별 → RAG만 적용 (γ=1.0)
cmd = [
    sys.executable, '../selective_rerun.py',
    '--baseline',           f'{WORK_DIR}/result_1_baseline.json',
    '--top_k_pct',          '5.0',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '4',
    '--cfg',                '1.0',
    '--save_path',          f'{WORK_DIR}/result_3_5pct_rag.json',
]
print('▶ Run 3: w/ 5% RAG (selective, γ=1.0, LoRA)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 4: w/ Expert-RAG — 5% + attn patch + embedding perturb, γ=1.0 (CFG bias 없음)
cmd = [
    sys.executable, '../selective_rerun.py',
    '--baseline',           f'{WORK_DIR}/result_1_baseline.json',
    '--top_k_pct',          '5.0',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '4',
    '--cfg',                '1.0',
    '--attn',               '3.0',
    '--perturb_weight',     '0.01',
    '--use_attn_patch',
    '--use_embed_perturb',
    '--save_path',          f'{WORK_DIR}/result_4_expert_rag.json',
]
print('▶ Run 4: w/ Expert-RAG (5% selective, attn+embed, γ=1.0, LoRA)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 5: w/ 5% Expert-CFG — 논문 메인 결과 (γ=1.5, attn+embed, 5% selective)
# 목표: Open=74.86%, Closed=91.18%, Overall=84.70%
cmd = [
    sys.executable, '../selective_rerun.py',
    '--baseline',           f'{WORK_DIR}/result_1_baseline.json',
    '--top_k_pct',          '5.0',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '4',
    '--cfg',                '1.5',
    '--attn',               '3.0',
    '--perturb_weight',     '0.01',
    '--use_attn_patch',
    '--use_embed_perturb',
    '--save_path',          f'{WORK_DIR}/result_5_5pct_expert_cfg.json',
]
print('▶ Run 5: w/ 5% Expert-CFG (γ=1.5, attn+embed, 5% selective, LoRA) ← 논문 메인')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

## Step 6. 결과 비교

In [ ]:
# ── 평가 함수 (팀원 방식: CLOSED=예측 토큰서 yes/no 탐색, OPEN=token recall) ──
import re as _re

def _tokenize(text):
    return _re.findall(r'[a-z0-9]+', str(text).lower())

def _score_answer(pred, gt, answer_type='OPEN'):
    gt = str(gt).strip().lower()
    pred_toks = _tokenize(pred)
    if gt in ('yes', 'no') or (answer_type or '').upper() == 'CLOSED':
        label = next((t for t in pred_toks if t in ('yes', 'no')), None)
        return float(label == gt)
    gt_toks = _tokenize(gt)
    return sum(t in pred_toks for t in gt_toks) / len(gt_toks) if gt_toks else 0.0

def rescore(results):
    closed = [r for r in results if r.get('answer_type', '').upper() == 'CLOSED']
    open_q = [r for r in results if r.get('answer_type', '').upper() == 'OPEN']
    sc = lambda r: _score_answer(r.get('prediction', ''), r.get('answer', ''), r.get('answer_type', 'OPEN'))
    c_acc  = sum(sc(r) for r in closed) / len(closed) * 100 if closed else float('nan')
    o_acc  = sum(sc(r) for r in open_q) / len(open_q) * 100 if open_q  else float('nan')
    ov_acc = sum(sc(r) for r in results) / len(results) * 100 if results else float('nan')
    return c_acc, o_acc, ov_acc, len(closed), len(open_q)

# ── 논문 Table 1 목표치 ──────────────────────────────────────────────────────
paper = {
    '1. baseline':     (90.44, 69.27, 82.04),
    '5. w/ 5% ExpCFG': (91.18, 74.86, 84.70),
}

runs = [
    ('1. baseline',       f'{WORK_DIR}/result_1_baseline.json',       '1.0', 'N', 'N'),
    ('2. w/ RAG',         f'{WORK_DIR}/result_2_rag_all.json',         '1.0', 'N', 'N'),
    ('3. w/ 5% RAG',      f'{WORK_DIR}/result_3_5pct_rag.json',        '1.0', 'N', '5%'),
    ('4. w/ Expert-RAG',  f'{WORK_DIR}/result_4_expert_rag.json',      '1.0', 'Y', '5%'),
    ('5. w/ 5% ExpCFG',   f'{WORK_DIR}/result_5_5pct_expert_cfg.json', '1.5', 'Y', '5%'),
]

print(f"{'Run':<22} {'Closed':>8} {'Open':>7} {'Overall':>8}  γ   Attn  Sel   nC   nO")
print('─' * 82)

for name, path, cfg, attn, sel in runs:
    if not os.path.exists(path):
        print(f"{name:<22}  (미완료)")
        continue
    with open(path) as f:
        data = json.load(f)
    results = data.get('results', data) if isinstance(data, dict) else data
    c_acc, o_acc, ov, nc, no_ = rescore(results)

    paper_mark = ''
    if name in paper:
        pc, po, pov = paper[name]
        dc, do_, dov = c_acc - pc, o_acc - po, ov - pov
        paper_mark = f'  ← paper C={pc:.2f}({dc:+.2f}) O={po:.2f}({do_:+.2f}) OA={pov:.2f}({dov:+.2f})'

    print(f"{name:<22} {c_acc:>7.2f}%  {o_acc:>6.2f}%  {ov:>7.2f}%  {cfg}   {attn}   {sel:>3}  {nc:>3}  {no_:>3}{paper_mark}")

In [ ]:
# ── 팀원 갭 진단 ─────────────────────────────────────────────────────────
import json, re

def load_results(path):
    try:
        with open(path) as f: d = json.load(f)
        return d.get('results', d) if isinstance(d, dict) else d
    except: return []

r1 = load_results(f'{WORK_DIR}/result_1_baseline.json')
if r1:
    toks = [len(r.get('token_preds', [])) for r in r1]
    open_q = [r for r in r1 if r.get('answer_type','').upper()=='OPEN']
    print('=== [1] token_preds 길이 분포 ===')
    print(f'  min={min(toks)}  max={max(toks)}  mean={sum(toks)/len(toks):.1f}')
    maxed = sum(1 for t in toks if t >= 32)
    print(f'  32토큰 한도에 걸린 샘플: {maxed}/{len(toks)} ({maxed/len(toks)*100:.1f}%)')
    print()

    print('=== [2] OPEN 오답 예시 (recall=0) ===')
    zeros = [(r['answer'], r['prediction']) for r in open_q
             if sum(t in re.findall(r"[a-z0-9]+", r['prediction'].lower())
                    for t in re.findall(r"[a-z0-9]+", r['answer'].lower())) == 0][:5]
    for gt, pred in zeros:
        print(f'  GT={gt!r:20} Pred={pred[:70]!r}')
    print()

# ── [3] highlights 필드 검증 ─────────────────────────────────────────────────
print('=== [3] test.json highlights 검증 ===')
INPUT_JSON = f'{REPO}/data/RadVQA/test.json'
with open(INPUT_JSON) as f:
    ann = json.load(f)

total = len(ann)
empty_hl   = sum(1 for a in ann if not a.get('highlights'))
empty_cap  = sum(1 for a in ann if not a.get('top_k_captions', [''])[0].strip())
print(f'  총 샘플: {total}')
print(f'  highlights 비어있는 샘플: {empty_hl} ({empty_hl/total*100:.1f}%)')
print(f'  top_k_captions 비어있는 샘플: {empty_cap} ({empty_cap/total*100:.1f}%)')
print()

# highlights 있는 샘플 예시
has_hl = [a for a in ann if a.get('highlights')]
if has_hl:
    ex = has_hl[0]
    print(f'  highlights 예시 (첫 샘플):')
    print(f'    type={type(ex["highlights"]).__name__}  len={len(ex["highlights"])}')
    print(f'    value={str(ex["highlights"])[:120]}')
else:
    print('  ⚠️  모든 샘플의 highlights가 비어있음 → Expert-Highlight 효과 없음!')
